## Prueba Big Data Zapping 2026
# Métricas de Negocio

### Dashboard

**Enlace:** _pendiente de publicación_

Dashboard en Streamlit con 7 páginas (resumen, flujo de clientes, reactivaciones, cambios de plan, ingresos, engagement y retención) y filtros globales por período, país, segmento, categoría, plan, canal, cupón y antigüedad. Lee las tablas que exporta la sección 8, así que sus cifras coinciden con las de este notebook.

Para ejecutarlo localmente, después de correr el notebook: `streamlit run dashboard/app.py`.

In [1]:
import pandas as pd

data_file_path = 'snapshot_subscriber.parquet'

In [2]:
df = pd.read_parquet(data_file_path)

In [3]:
print(df.info(max_cols=len(df.columns)))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1688997 entries, 0 to 1688996
Data columns (total 61 columns):
 #   Column                      Non-Null Count    Dtype  
---  ------                      --------------    -----  
 0   id                          1688997 non-null  int64  
 1   country_iso                 1688997 non-null  object 
 2   date                        1688997 non-null  object 
 3   created                     1688997 non-null  object 
 4   updated                     1688997 non-null  object 
 5   plan_status                 1688997 non-null  object 
 6   is_b2b                      1688997 non-null  int64  
 7   is_dcb                      1688997 non-null  int64  
 8   category                    1688997 non-null  object 
 9   sub_category                1688997 non-null  object 
 10  is_subscriber               1688997 non-null  int64  
 11  is_mrr_contributor          1688997 non-null  int64  
 12  time_watched                1688997 non-null  int64  
 1

In [4]:
df['time_watched'].head(5)

0    5014642
1    5014642
2    5014642
3    5002661
4    5002661
Name: time_watched, dtype: int64

In [5]:
# Quiero ver si hay algún día faltante y el rango de fechas con el que trabajo
dates = pd.to_datetime(df['date'])
dates = dates.dt.normalize()
dates_unique = dates.unique()
dates_range = pd.date_range(start=dates_unique.min(), end=dates_unique.max(), freq='D')

print(len(dates_range.difference(dates)), 'días faltantes entre', df['date'].min(), 'y', df['date'].max())

0 días faltantes entre 2026-01-01 03:30:00 y 2026-09-07 03:30:00


In [6]:
# Usuarios únicos y por país
print(df['id'].nunique(), 'usuarios únicos')

ids_by_country = df.groupby('country_iso')['id'].nunique()

print(ids_by_country)
print(ids_by_country.sum())

10001 usuarios únicos
country_iso
BR     992
CL    2951
EC    5951
PE     107
Name: id, dtype: int64
10001


### **Supuestos**

1. Se entiende que si un usuario-plan no aparece un día, ese día no lo tiene activo. Esto implica que un salto de 1 día genera una baja y luego una alta (reactivación). La intermitencia de planes (alta → baja → alta) es un comportamiento propio del rubro, por lo que todo salto cuenta como baja y reactivación, sin tolerancia, también en la vida media y el churn. La columna `gap_days` registra la duración de cada salto.
2. Existen días en el registro que reportan pérdidas masivas de usuarios. Esto impacta fuertemente las métricas y se desconoce el origen de estas 'bajas'. Para identificar estos sucesos se usan umbrales derivados de los datos. Un volumen es atípico si supera la *Tukey fence for outliers* (`MASS_TUKEY_K = 3`), calculada sobre el log10 de los conteos por (día, grupo). Una baja masiva exige además que el partner (o plan) pierda al menos `MASS_CHURN_MIN_SHARE` (50%) de su base activa ese día. Una migración masiva solo exige el volumen atípico.
3. Los usuarios presentes el primer día del snapshot (`DAY0`) ya estaban activos, así que no se cuentan como alta para ese día. Los del último día (`DAYN`) siguen activos y no se cuentan como baja.
4. Si un usuario tiene 2 planes base el mismo día, se considera vigente el de `created` más reciente, es decir el plan nuevo.
5. El enunciado menciona un campo `created_user`, pero no existe. Se usa como proxy la fecha mínima de `created` (creación de plan) de cada usuario para cualquier cálculo relevante. El equipo de Zapping validó este proxy.
6. Para el MRR, solo suman las filas con `is_mrr_contributor = 1`. El precio mensual es `price_paid`, salvo en planes multi-período (`periods_in_months > 1`) donde `price_paid > original_price`, indicando un registro del cobro del período completo. Este último se divide por `periods_in_months`. `original_price` corresponde a un precio de lista mensual.
7. Se convierten todas las monedas a USD con con las tasas fijas a la fecha (26-10-2026). El MRR se reporta en moneda local por país.
8. Para los upgrades y downgrades, se compara el precio de lista mensual convertido a USD del plan base anterior y del nuevo. Si sube es upgrade, si baja es downgrade y si es igual (+-1%) se asume migración. Solo cuenta un cambio entre días consecutivos y un cambio tras un salto queda registrado como baja + reactivación con `mismo_plan = False`.
9. `time_watched` son los segundos de consumo de los últimos 30 días (ventana móvil, confirmado por Zapping). Para el MAU, un usuario es activo en el mes si tiene `time_watched > 0` en el snapshot de fin de mes, es decir, si consumió contenido en los 30 días previos.

In [7]:
import numpy as np
from pathlib import Path
from IPython.display import display
from lifelines import KaplanMeierFitter
from lifelines.exceptions import StatError
from lifelines.utils import restricted_mean_survival_time

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

FX_TO_USD = {'USD': 1.0, 'CLP': 1 / 991.40, 'BRL': 1 / 5.02, 'PEN': 1 / 3.45}
MASS_TUKEY_K = 3                  # volumen atípico: sobre la cerca de Tukey extrema (Q3 + k·IQR) del log10 de conteos por (día, grupo)
MASS_CHURN_MIN_SHARE = 0.5        # baja masiva: además, el partner/plan pierde >= 50% de su base activa ese día
TOP_N = 8                         # categorías a mostrar en cohortes de alta cardinalidad (resto -> 'Otros')
OUT_DIR = Path('outputs')
OUT_DIR.mkdir(exist_ok=True)

### 1. Limpieza y variables derivadas

In [8]:
def to_datetime_fast(s):
    """Parsea fechas con formatos mixtos usando solo los valores únicos. Devuelve fechas UTC sin zona."""
    uniq = s.dropna().unique()
    parsed = pd.to_datetime(pd.Series(uniq), utc=True, format='mixed', errors='coerce').dt.tz_localize(None)
    return pd.to_datetime(s.map(dict(zip(uniq, parsed))))


# Los nulos vienen como el string 'nan' en las columnas de texto.
# Se pasan a NaN.
obj_cols = df.select_dtypes('object').columns
df[obj_cols] = df[obj_cols].replace('nan', np.nan)

df['d'] = pd.to_datetime(df['date']).dt.normalize()
df['month'] = df['d'].dt.to_period('M')     # Unidad de tiempo común
for col in ['created', 'updated', 'payment_timestamp', 'next_billing_date']:
    df[f'{col}_dt'] = to_datetime_fast(df[col])

DAY0, DAYN = df['d'].min(), df['d'].max()

# Precio mensual normalizado y conversión a USD
periods = df['periods_in_months'].fillna(1).clip(lower=1)
full_period_charge = (periods > 1) & (df['price_paid'] > df['original_price'])
df['price_month'] = np.where(full_period_charge, df['price_paid'] / periods, df['price_paid'])
fx = df['currency'].map(FX_TO_USD)
assert fx.notna().all(), 'Hay monedas sin tasa de cambio definida'
df['mrr_local'] = df['price_month'] * df['is_mrr_contributor']
df['mrr_usd'] = df['mrr_local'] * fx
df['list_price_usd'] = df['original_price'] * fx
df['has_coupon'] = df['coupon_type'].notna()

# Proxy de fecha de creación del usuario (a falta de user_created)
df['user_created'] = df.groupby('id')['created_dt'].transform('min')

print(f'Ventana: {DAY0.date()} -> {DAYN.date()} ({(DAYN - DAY0).days + 1} días)')
print(f'Cobros de período completo normalizados: {full_period_charge.sum():,} filas')

Ventana: 2026-01-01 -> 2026-09-07 (250 días)
Cobros de período completo normalizados: 250 filas


### 2. Tablas base

- **`user_day`**: un registro por usuario-día. Trae los atributos del plan principal (el plan base vigente, o un addon si ese día no tiene base), el MRR total del día (base + addons), la cantidad de addons, si vio contenido y su antigüedad.
- **`base_day`**: el subconjunto de `user_day` con plan base (un plan base por usuario-día, supuesto 4).
- **`build_spells`**: agrupa días consecutivos de presencia en *spells* (períodos continuos). Con `tol > 0` se toleran saltos de hasta `tol` días.

In [9]:
PLAN_COLS = ['package_id', 'package_name', 'package_type', 'package_category', 'category', 'sub_category',
             'country_iso', 'currency', 'original_price', 'price_paid', 'price_month', 'mrr_usd',
             'list_price_usd', 'periods_in_months', 'coupon_type', 'has_coupon', 'acquisition', 'is_b2b', 'partner_id',
             'cancel_at_period_end', 'created_dt']

# Fila principal por usuario-día: base antes que addon; dentro de base, el plan creado más recientemente
primary = (df.sort_values(['id', 'd', 'main', 'created_dt'], ascending=[True, True, False, False])
             .drop_duplicates(['id', 'd']))

df['is_addon'] = (df['main'] == 0).astype(int)
day_agg = df.groupby(['id', 'd']).agg(mrr_usd_total=('mrr_usd', 'sum'),
                                      mrr_local_total=('mrr_local', 'sum'),
                                      n_packages=('package_id', 'size'),
                                      n_addons=('is_addon', 'sum'),
                                      time_watched=('time_watched', 'max')).reset_index()
addons_by_day = (df[df['main'] == 0].groupby(['id', 'd'])['package_name']
                 .agg(lambda s: ', '.join(sorted(s))).rename('addons').reset_index())

user_day = (primary[['id', 'd', 'month', 'main', 'user_created'] + PLAN_COLS]
            .rename(columns={'main': 'has_base'})
            .merge(day_agg, on=['id', 'd'], how='left')
            .merge(addons_by_day, on=['id', 'd'], how='left'))

user_day['is_paying'] = user_day['mrr_usd_total'] > 0
user_day['watched'] = user_day['time_watched'] > 0
user_day['tenure_days'] = (user_day['d'] - user_day['user_created'].dt.normalize()).dt.days.clip(lower=0)
user_day['tenure_bucket'] = pd.cut(user_day['tenure_days'], bins=[-1, 30, 90, 180, 365, 730, np.inf],
                                   labels=['<1M', '1-3M', '3-6M', '6-12M', '1-2A', '>2A'])

# Etiquetas de cohorte
user_day['segmento'] = np.where(user_day['is_b2b'] == 1, 'B2B', 'B2C')
user_day['coupon_label'] = np.where(user_day['has_coupon'], 'Con cupón', 'Sin cupón')
user_day['category'] = user_day['category'].fillna('SIN_CATEGORIA')
# El mismo nombre de plan existe en varios países/monedas (y con espacios distintos): se etiqueta con el país
user_day['plan_label'] = (user_day['package_name'].str.normalize('NFKC').str.strip()
                          + ' (' + user_day['country_iso'] + ')')
# Grupo para detectar bajas masivas: el partner B2B si existe; si no, el plan
user_day['mass_group'] = ('partner_' + user_day['partner_id'].astype('Int64').astype('string')).fillna(
    'plan_' + user_day['plan_label'])
for src, dst in [('plan_label', 'package_grp'), ('acquisition', 'acquisition_grp')]:
    top = user_day.groupby(src)['id'].nunique().nlargest(TOP_N).index
    user_day[dst] = user_day[src].where(user_day[src].isin(top), 'Otros').fillna('Otros')

base_day = user_day[user_day['has_base'] == 1].copy()

print(f'user_day: {len(user_day):,} filas | base_day: {len(base_day):,} filas | '
      f'usuario-días sin plan base: {(user_day["has_base"] == 0).sum():,}')

user_day: 1,631,814 filas | base_day: 1,631,438 filas | usuario-días sin plan base: 376


In [10]:
def build_spells(frame, keys, tol=0):
    """Agrupa días de presencia consecutivos en spells por `keys`.

    Abre un spell nuevo cuando faltan más de `tol` días entre dos registros.
    gap_before = días sin registro antes del spell (NaN si es el primero observado).
    """
    f = frame[keys + ['d']].drop_duplicates().sort_values(keys + ['d'])
    diff = f.groupby(keys)['d'].diff().dt.days
    new_spell = diff.isna() | (diff - 1 > tol)
    f['spell_id'] = new_spell.cumsum()
    f['gap_before'] = np.where(new_spell, diff - 1, np.nan)
    sp = (f.groupby('spell_id')
            .agg(**{k: (k, 'first') for k in keys},
                 start=('d', 'min'), end=('d', 'max'),
                 gap_before=('gap_before', 'first'), days_present=('d', 'size'))
            .reset_index(drop=True))
    sp['spell_n'] = sp.groupby(keys).cumcount() + 1
    sp['prev_end'] = sp.groupby(keys)['end'].shift()
    sp['gap_after'] = sp.groupby(keys)['gap_before'].shift(-1)
    sp['left_censored'] = sp['start'] == DAY0
    sp['right_censored'] = sp['end'] == DAYN
    return sp


def tukey_fence_log(counts, k=MASS_TUKEY_K):
    """Umbral de volumen atípico: cerca superior de Tukey (Q3 + k·IQR) sobre log10 de los conteos.

    Los conteos por (día, grupo) tienen cola muy pesada (mediana 1, máximo en miles), por eso se usa escala log.
    """
    log_counts = np.log10(counts)
    q1, q3 = log_counts.quantile([0.25, 0.75])
    return 10 ** (q3 + k * (q3 - q1))


sp_user = build_spells(user_day, ['id'], tol=0)
print(f'{len(sp_user):,} spells estrictos de servicio para {sp_user["id"].nunique():,} usuarios')

13,030 spells estrictos de servicio para 10,001 usuarios


### 3. Actividad 1: altas y bajas diarias → `df_altas_bajas`

- **`alta`**: el usuario empieza un período de servicio (no tenía ningún plan el día anterior). Se informa el plan contratado.
- **`baja`**: el usuario deja de tener cualquier plan. `date` es el **primer día sin plan** y `last_active_date` es su último día activo. Se informa el plan cancelado.
- **`alta_addon` / `baja_addon`**: el usuario contrata o cancela un addon **mientras sigue activo**. Los addons que llegan o se van junto con el servicio aparecen en la columna `addons` del evento de servicio.
- `es_primera_alta = False` indica que el alta es una **reactivación** (ver actividad 2).
- `gap_days`: en una alta, los días que el usuario estuvo fuera antes de volver; en una baja, los días que tardará en volver (NaN si no vuelve dentro de la ventana).
- **Intermitencia.** Salir y volver es parte del comportamiento del rubro (supuesto 1): cada salto, aunque sea de 1 día, genera una baja y una reactivación. En una baja, `gap_days` no nulo indica que el usuario volvió dentro de la ventana.
- `baja_masiva = True` marca las bajas de un (día, grupo) donde el grupo (partner B2B o, si no hay, plan) **pierde al menos el 50% de su base activa** y el volumen es atípico (sobre la cerca de Tukey extrema). Típicamente corresponde al término de un convenio B2B. `bajas_grupo_dia` y `pct_base_grupo` quedan en la tabla como evidencia. Las bajas periódicas de un partner por su ciclo de facturación (p.ej. partner 1271, ~5–11% de su base cada mes) **no** son masivas: son churn recurrente.

In [11]:
INFO_COLS = PLAN_COLS + ['mrr_usd_total', 'n_addons', 'addons', 'user_created', 'mass_group']
ud_info = user_day[['id', 'd'] + INFO_COLS]

# --- Altas de servicio ---
altas = (sp_user[~sp_user['left_censored']]
         .merge(ud_info, left_on=['id', 'start'], right_on=['id', 'd'], how='left'))
altas['date'] = altas['start']
altas['event_type'] = 'alta'
altas['gap_days'] = altas['gap_before']
created_lag = (altas['start'] - altas['user_created'].dt.normalize()).dt.days
altas['origen_reactivacion'] = np.select(
    [altas['spell_n'] > 1, created_lag > 2],          # historial dentro de la ventana / previo a la ventana
    ['dentro_ventana', 'previo_ventana'], default=None)
altas['origen_reactivacion'] = altas['origen_reactivacion'].replace({'None': np.nan})
altas['es_primera_alta'] = altas['origen_reactivacion'].isna()

# --- Bajas de servicio ---
bajas = (sp_user[~sp_user['right_censored']]
         .merge(ud_info, left_on=['id', 'end'], right_on=['id', 'd'], how='left'))
bajas['date'] = bajas['end'] + pd.Timedelta(days=1)
bajas['last_active_date'] = bajas['end']
bajas['event_type'] = 'baja'
bajas['gap_days'] = bajas['gap_after']
# Baja masiva: el grupo (partner o plan) pierde >= MASS_CHURN_MIN_SHARE de su base activa, con volumen atípico
group_base = user_day.groupby(['d', 'mass_group'])['id'].nunique().rename('base_grupo')
bajas = bajas.merge(group_base, left_on=['end', 'mass_group'], right_index=True, how='left')
bajas['bajas_grupo_dia'] = bajas.groupby(['date', 'mass_group'])['id'].transform('size')
bajas['pct_base_grupo'] = bajas['bajas_grupo_dia'] / bajas['base_grupo']
CHURN_FENCE = tukey_fence_log(bajas.groupby(['date', 'mass_group']).size())
bajas['baja_masiva'] = (bajas['pct_base_grupo'] >= MASS_CHURN_MIN_SHARE) & (bajas['bajas_grupo_dia'] > CHURN_FENCE)

# --- Altas y bajas de addons con el usuario activo ---
addon_rows = df.loc[df['main'] == 0, ['id', 'd'] + PLAN_COLS]
sp_addon = build_spells(addon_rows, ['id', 'package_id'], tol=0)
active_idx = pd.MultiIndex.from_frame(user_day[['id', 'd']])

a_add = sp_addon[~sp_addon['left_censored']].copy()
a_add = a_add[pd.MultiIndex.from_arrays([a_add['id'], a_add['start'] - pd.Timedelta(days=1)]).isin(active_idx)]
a_add = a_add.merge(addon_rows, left_on=['id', 'package_id', 'start'], right_on=['id', 'package_id', 'd'])
a_add['date'], a_add['event_type'], a_add['gap_days'] = a_add['start'], 'alta_addon', a_add['gap_before']

b_add = sp_addon[~sp_addon['right_censored']].copy()
b_add = b_add[pd.MultiIndex.from_arrays([b_add['id'], b_add['end'] + pd.Timedelta(days=1)]).isin(active_idx)]
b_add = b_add.merge(addon_rows, left_on=['id', 'package_id', 'end'], right_on=['id', 'package_id', 'd'])
b_add['date'], b_add['event_type'], b_add['gap_days'] = b_add['end'] + pd.Timedelta(days=1), 'baja_addon', b_add['gap_after']
b_add['last_active_date'] = b_add['end']

EVENT_COLS = (['date', 'id', 'event_type', 'es_primera_alta', 'origen_reactivacion', 'gap_days',
               'last_active_date',
               'baja_masiva', 'bajas_grupo_dia', 'pct_base_grupo']
              + PLAN_COLS + ['mrr_usd_total', 'n_addons', 'addons'])
df_altas_bajas = (pd.concat([altas, bajas, a_add, b_add], ignore_index=True)
                  .reindex(columns=EVENT_COLS)
                  .rename(columns={'created_dt': 'plan_created'})
                  .sort_values(['date', 'id', 'event_type'])
                  .reset_index(drop=True))

display(df_altas_bajas['event_type'].value_counts().rename('eventos').to_frame())
# Evidencia del criterio: (día, grupo) con más bajas, su volumen y la fracción de la base perdida
print(f'Volumen atípico de bajas por (día, grupo): > {CHURN_FENCE:.1f} usuarios (cerca de Tukey, k={MASS_TUKEY_K})')
display(bajas.groupby(['date', 'mass_group', 'country_iso'])
        .agg(bajas=('id', 'size'), base_grupo=('base_grupo', 'first'), pct_base_grupo=('pct_base_grupo', 'first'),
             baja_masiva=('baja_masiva', 'first'))
        .sort_values('bajas', ascending=False).head(12).round(3))
display(df_altas_bajas.head(10))

,eventos
event_type,
baja,7731
alta,6315
alta_addon,235
baja_addon,169


Volumen atípico de bajas por (día, grupo): > 16.0 usuarios (cerca de Tukey, k=3)


bajas  base_grupo  pct_base_grupo  baja_masiva
date       mass_group   country_iso                                                
2026-06-12 partner_1351 EC            2359        2535           0.931         True
           partner_2181 EC             262         262           1.000         True
2026-01-12 partner_1271 EC             135        1181           0.114        False
2026-02-11 partner_1271 EC             127        1232           0.103        False
2026-08-11 partner_1271 EC             120        1326           0.090        False
2026-06-11 partner_1271 EC             111        1318           0.084        False
2026-05-11 partner_1271 EC              91        1313           0.069        False
2026-03-11 partner_1271 EC              87        1233           0.071        False
2026-07-10 partner_1271 EC              86        1388           0.062        False
2026-04-11 partner_1271 EC              75        1277           0.059        False
2026-08-10 partner_1271 EC              74        1398           0.053        False
2026-03-12 partner_1271 EC              72        1170           0.062        False

,date,id,event_type,es_primera_alta,origen_reactivacion,gap_days,last_active_date,baja_masiva,bajas_grupo_dia,pct_base_grupo,package_id,package_name,package_type,package_category,category,sub_category,country_iso,currency,original_price,price_paid,price_month,mrr_usd,list_price_usd,periods_in_months,coupon_type,has_coupon,acquisition,is_b2b,partner_id,cancel_at_period_end,plan_created,mrr_usd_total,n_addons,addons
0,2026-01-02,5176193,baja,NaN,NaN,42.0,2026-01-01,False,3.0,0.012146,671,Futbol,base,b2c,B2C,monthly,CL,CLP,18900,18900.00,18900.00,19.063950,19.063950,1.0,NaN,False,www,0,NaN,0,2025-05-26 04:40:18,19.063950,0.0,NaN
1,2026-01-02,6260507,baja,NaN,NaN,NaN,2026-01-01,False,1.0,0.500000,681,TNT Sports HD,addon,NaN,B2C,monthly,CL,CLP,13500,13500.00,13500.00,13.617107,13.617107,NaN,NaN,False,NaN,0,NaN,1,2024-03-02 16:08:27,13.617107,1.0,TNT Sports HD
2,2026-01-02,6551743,baja,NaN,NaN,1.0,2026-01-01,False,3.0,0.012146,671,Futbol,base,b2c,B2C,monthly,CL,CLP,18900,18900.00,18900.00,19.063950,19.063950,1.0,NaN,False,NaN,0,NaN,0,2025-03-01 21:52:51,19.063950,0.0,NaN
3,2026-01-02,10807022171,baja,NaN,NaN,50.0,2026-01-01,False,2.0,0.024096,341,Premium,base,b2c,B2C,monthly,EC,USD,28,27.99,27.99,27.990000,28.000000,1.0,NaN,False,b2c,0,NaN,1,2025-02-01 20:48:10,27.990000,0.0,NaN
4,2026-01-02,10810108441,alta,False,previo_ventana,NaN,NaT,NaN,NaN,NaN,571,Plus,base,b2c,B2C,monthly,CL,CLP,19900,19900.00,19900.00,20.072625,20.072625,1.0,NaN,False,NaN,0,NaN,0,2024-12-18 18:13:13,20.072625,0.0,NaN
5,2026-01-02,10854332951,baja,NaN,NaN,NaN,2026-01-01,False,2.0,0.024096,341,Premium,base,b2c,B2C,monthly,EC,USD,28,27.99,27.99,27.990000,28.000000,1.0,NaN,False,b2c,0,NaN,0,2025-02-02 00:26:33,27.990000,0.0,NaN
6,2026-01-02,10854617961,baja,NaN,NaN,NaN,2026-01-01,False,3.0,0.012146,671,Futbol,base,b2c,COUPON,1-UCH-COQ2,CL,CLP,18900,16900.00,16900.00,0.000000,19.063950,1.0,percentage,True,ABONADOS-UCH-10MESES-FUTBOL,0,NaN,0,2025-12-04 00:36:03,0.000000,0.0,NaN
7,2026-01-02,10900193051,baja,NaN,NaN,NaN,2026-01-01,False,3.0,0.050847,551,Nacional,base,hardbundle,HARDBUNDLE,MASKED_B2B_PARTNER,CL,CLP,7900,7900.00,7900.00,7.968529,7.968529,1.0,NaN,False,hardbundle,1,142.0,0,2025-07-03 00:37:25,7.968529,0.0,NaN
8,2026-01-02,10908416651,baja,NaN,NaN,3.0,2026-01-01,False,2.0,0.003311,551,Nacional,base,b2c,B2C,monthly,CL,CLP,7900,7900.00,7900.00,7.968529,7.968529,1.0,NaN,False,CMR-20DCTO-4MESES,0,NaN,0,2025-08-30 14:59:53,7.968529,0.0,NaN
9,2026-01-02,10909714981,baja,NaN,NaN,NaN,2026-01-01,False,3.0,0.050847,551,Nacional,base,hardbundle,HARDBUNDLE,MASKED_B2B_PARTNER,CL,CLP,7900,7900.00,7900.00,7.968529,7.968529,1.0,NaN,False,hardbundle,1,142.0,0,2025-09-05 18:32:37,7.968529,0.0,NaN


In [12]:
# Resumen diario: eventos por día + usuarios activos
daily_events = (df_altas_bajas.pivot_table(index='date', columns='event_type', values='id', aggfunc='count')
                .reindex(pd.date_range(DAY0, DAYN, freq='D')).fillna(0).astype(int))
daily_events['usuarios_activos'] = user_day.groupby('d')['id'].nunique()
daily_events['altas_netas'] = daily_events['alta'] - daily_events['baja']
daily_events.index.name = 'date'

# Resumen mensual por país
monthly_events = (df_altas_bajas.assign(month=df_altas_bajas['date'].dt.to_period('M'))
                  .pivot_table(index=['month', 'country_iso'], columns='event_type', values='id', aggfunc='count')
                  .fillna(0).astype(int))
display(daily_events.tail(10))
display(monthly_events.unstack('country_iso').loc[:, ['alta', 'baja']])

# Bajas orgánicas (sin fin de convenio) por mes y país
display(df_altas_bajas[(df_altas_bajas['event_type'] == 'baja') & (df_altas_bajas['baja_masiva'] != True)]
       .assign(month=lambda x: x['date'].dt.to_period('M'))
       .pivot_table(index='month', columns='country_iso', values='id', aggfunc='count', fill_value=0)
       .rename_axis(columns='bajas orgánicas'))

event_type,alta,alta_addon,baja,baja_addon,usuarios_activos,altas_netas
date,,,,,,
2026-08-29,29,0,8,0,5272,21
2026-08-30,18,1,19,0,5271,-1
2026-08-31,15,0,25,0,5261,-10
2026-09-01,19,0,11,2,5269,8
2026-09-02,29,1,13,1,5285,16
2026-09-03,16,2,9,0,5292,7
2026-09-04,16,0,20,1,5288,-4
2026-09-05,17,1,6,0,5299,11
2026-09-06,20,2,27,1,5292,-7


event_type    alta                      baja                     
country_iso     BR     CL     EC    PE    BR     CL      EC    PE
month                                                            
2026-01      121.0  212.0  323.0  10.0  38.0  202.0   352.0  12.0
2026-02       97.0  265.0  431.0  17.0  39.0  161.0   225.0   9.0
2026-03      119.0  290.0  578.0  11.0  81.0  212.0   382.0  15.0
2026-04      115.0  195.0  489.0  18.0  56.0  194.0   368.0  11.0
2026-05      107.0  200.0  316.0   9.0  70.0  192.0   328.0   8.0
2026-06      116.0  203.0  332.0   1.0  64.0  175.0  2998.0  40.0
2026-07       98.0  235.0  507.0   0.0  60.0  210.0   377.0  20.0
2026-08      114.0  267.0  384.0   NaN  64.0  228.0   443.0   NaN
2026-09       26.0   34.0   75.0   NaN  20.0   48.0    29.0   NaN

bajas orgánicas,BR,CL,EC,PE
month,,,,
2026-01,38,202,352,12
2026-02,39,161,225,9
2026-03,81,212,382,15
2026-04,56,194,368,11
2026-05,70,192,328,8
2026-06,64,175,377,40
2026-07,60,210,377,20
2026-08,64,228,443,0
2026-09,20,48,29,0


### 4. Actividad 2: reactivaciones → `df_reactivaciones`

Una **reactivación** es un *alta* de un usuario que ya había tenido plan antes:
- `origen = 'dentro_ventana'`: el usuario tuvo un spell anterior dentro de los datos. Se informa su **último plan** (`last_*`) y cuántos días estuvo fuera (`gap_days`).
- `origen = 'previo_ventana'`: aparece por primera vez en los datos, pero su `created` es más de 2 días anterior a esa aparición, así que tenía historial antes del 2026-01-01. Su último plan **no es observable** y los campos `last_*` quedan vacíos.
- `gap_days`: días sin plan antes de volver. No hay umbral para reactivaciones "técnicas": la intermitencia es parte del negocio (supuesto 1) y se describe más abajo.

In [13]:
REACT_PLAN_COLS = ['package_id', 'package_name', 'package_category', 'category', 'currency',
                   'original_price', 'price_paid', 'mrr_usd', 'country_iso']

react = altas[altas['origen_reactivacion'].notna()].copy()
last_plan = (ud_info[['id', 'd'] + REACT_PLAN_COLS]
             .rename(columns={c: f'last_{c}' for c in REACT_PLAN_COLS}).rename(columns={'d': 'last_active_date'}))
react = react.rename(columns={c: f'new_{c}' for c in REACT_PLAN_COLS})
react = react.merge(last_plan, left_on=['id', 'prev_end'], right_on=['id', 'last_active_date'], how='left')

react['mismo_plan'] = np.where(react['last_package_id'].notna(),
                               react['last_package_id'] == react['new_package_id'], np.nan)
react['delta_mrr_usd'] = react['new_mrr_usd'] - react['last_mrr_usd']

df_reactivaciones = (react[['date', 'id', 'origen_reactivacion', 'gap_days', 'mismo_plan', 'last_active_date']
                           + [f'last_{c}' for c in REACT_PLAN_COLS] + [f'new_{c}' for c in REACT_PLAN_COLS]
                           + ['delta_mrr_usd', 'user_created']]
                     .rename(columns={'origen_reactivacion': 'origen'})
                     .sort_values(['date', 'id']).reset_index(drop=True))

print(f'{len(df_reactivaciones):,} reactivaciones de {df_reactivaciones["id"].nunique():,} usuarios')
display(df_reactivaciones['origen'].value_counts().rename('reactivaciones').to_frame())
display(df_reactivaciones.head(10))

3,183 reactivaciones de 1,765 usuarios


,reactivaciones
origen,
dentro_ventana,3029
previo_ventana,154


,date,id,origen,gap_days,mismo_plan,last_active_date,last_package_id,last_package_name,last_package_category,last_category,last_currency,last_original_price,last_price_paid,last_mrr_usd,last_country_iso,new_package_id,new_package_name,new_package_category,new_category,new_currency,new_original_price,new_price_paid,new_mrr_usd,new_country_iso,delta_mrr_usd,user_created
0,2026-01-02,10810108441,previo_ventana,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,571,Plus,b2c,B2C,CLP,19900,19900.0,20.072625,CL,NaN,2024-12-18 18:13:13
1,2026-01-03,6551743,dentro_ventana,1.0,1.0,2026-01-01,671.0,Futbol,b2c,B2C,CLP,18900.0,18900.0,19.063950,CL,671,Futbol,b2c-yearly,B2C,CLP,18900,18900.0,19.063950,CL,0.000000,2025-03-01 21:52:51
2,2026-01-03,31408762,previo_ventana,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,551,Nacional,b2c,B2C,CLP,7900,7900.0,7.968529,CL,NaN,2025-01-17 15:17:07
3,2026-01-03,32240142,previo_ventana,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,571,Plus,b2c,B2C,CLP,19900,19900.0,20.072625,CL,NaN,2024-11-22 17:27:20
4,2026-01-03,10772461051,previo_ventana,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,551,Nacional,b2c,B2C,CLP,7900,7900.0,7.968529,CL,NaN,2025-11-12 15:51:23
5,2026-01-03,10819433691,previo_ventana,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,571,Plus,b2c,B2C,CLP,19900,19900.0,20.072625,CL,NaN,2024-12-10 03:18:54
6,2026-01-03,10842856321,previo_ventana,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,122,Zapping Lite Plus,hardbundle,HARDBUNDLE,BRL,20,19.9,3.964143,BR,NaN,2024-12-21 14:26:02
7,2026-01-03,10888817181,previo_ventana,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,571,Plus,b2c,B2C,CLP,19900,19900.0,20.072625,CL,NaN,2025-05-03 01:35:22
8,2026-01-03,10907192681,previo_ventana,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,551,Nacional,b2c,B2C,CLP,7900,7900.0,7.968529,CL,NaN,2025-08-22 02:17:02
9,2026-01-03,10913546941,dentro_ventana,1.0,0.0,2026-01-01,571.0,Plus,b2c,B2C,CLP,19900.0,19900.0,20.072625,CL,551,Nacional,b2c,B2C,CLP,7900,7900.0,7.968529,CL,-12.104095,2025-10-01 21:22:21


#### Intermitencia
Cuántas veces sale y vuelve cada usuario dentro de la ventana, qué fracción de las bajas orgánicas vuelve y cuánto dura el salto. Las bajas de los últimos días tienen menos tiempo para volver, así que la tasa de retorno es una cota inferior.

In [14]:
# Reactivaciones dentro de la ventana por usuario (0 = nunca se fue, o se fue y no volvió)
ciclos = sp_user.groupby('id').size().sub(1)
dist_ciclos = ciclos.clip(upper=5).value_counts().sort_index().rename(index={5: '5+'}).rename('usuarios').to_frame()
dist_ciclos['pct'] = (dist_ciclos['usuarios'] / dist_ciclos['usuarios'].sum()).round(3)
display(dist_ciclos.rename_axis('reactivaciones_en_ventana'))

bajas_org = bajas[~bajas['baja_masiva']]
print(f"Bajas orgánicas que vuelven dentro de la ventana: {bajas_org['gap_after'].notna().mean():.1%} "
      f"de {len(bajas_org):,}")
display(df_reactivaciones.loc[df_reactivaciones['origen'] == 'dentro_ventana', 'gap_days']
       .describe(percentiles=[0.25, 0.5, 0.75, 0.9]).round(1).rename('días sin plan').to_frame().T)

,usuarios,pct
reactivaciones_en_ventana,,
0,8350,0.835
1,978,0.098
2,328,0.033
3,170,0.017
4,73,0.007
5+,102,0.010


Bajas orgánicas que vuelven dentro de la ventana: 58.7% de 5,110


,count,mean,std,min,25%,50%,75%,90%,max
días sin plan,3029.0,10.5,21.6,1.0,1.0,2.0,9.0,29.0,204.0


### 5. Actividad 3: upgrade / downgrade → `df_cambios_plan`

Hay cambio de plan cuando el plan base vigente (`package_id`) del día `d` es distinto al del día `d-1`, con el usuario activo ambos días. Se clasifica según el precio de lista mensual en USD (supuesto 8): `upgrade`, `downgrade` o `lateral`. Si el plan conserva el nombre y solo cambia de `package_id` (p.ej. *Zapping Full* de 18.900 a 19.900 CLP), es un **`reajuste_precio`** del catálogo y no una decisión del usuario.

`migracion_masiva = True` marca los pares (plan anterior → plan nuevo) cuyo volumen en un día es **atípico**: supera la cerca de Tukey extrema del log10 de los conteos por (día, par). En un día típico un par de planes registra 1–2 cambios, así que un volumen así no es una suma de decisiones individuales sino una migración administrativa (cambio de catálogo, de versión o campaña). Conviene separarlas al analizar tasas. Aquí no se exige un % de la base, porque las migraciones de catálogo se ejecutan en tandas de varios días (p.ej. Básico→Demo EC: 11%, 9%, 16% y 30% de la base en 4 días).

In [15]:
CHANGE_COLS = ['package_id', 'package_name', 'package_category', 'category', 'currency',
               'original_price', 'price_paid', 'mrr_usd', 'list_price_usd']

b = base_day[['id', 'd', 'country_iso'] + CHANGE_COLS].sort_values(['id', 'd'])
prev = b.groupby('id')[['d'] + CHANGE_COLS].shift()
prev.columns = [f'prev_{c}' for c in prev.columns]
cmb = pd.concat([b, prev], axis=1)
cmb = cmb[((cmb['d'] - cmb['prev_d']).dt.days == 1) & (cmb['package_id'] != cmb['prev_package_id'])].copy()

delta_list = cmb['list_price_usd'] - cmb['prev_list_price_usd']
tol_price = 0.01 * cmb[['list_price_usd', 'prev_list_price_usd']].max(axis=1)
cmb['same_name'] = cmb['package_name'] == cmb['prev_package_name']
cmb['tipo'] = np.select([cmb['same_name'], delta_list > tol_price, delta_list < -tol_price],
                        ['reajuste_precio', 'upgrade', 'downgrade'], default='lateral')
cmb['delta_mrr_usd'] = cmb['mrr_usd'] - cmb['prev_mrr_usd']
cmb['cross_currency'] = cmb['currency'] != cmb['prev_currency']
pair_size = cmb.groupby(['d', 'prev_package_id', 'package_id'])['id'].transform('size')
MIGRATION_FENCE = tukey_fence_log(cmb.groupby(['d', 'prev_package_id', 'package_id']).size())
cmb['migracion_masiva'] = pair_size > MIGRATION_FENCE

df_cambios_plan = (cmb.rename(columns={'d': 'date', **{c: f'new_{c}' for c in CHANGE_COLS}})
                   [['date', 'id', 'country_iso', 'tipo', 'migracion_masiva', 'same_name', 'cross_currency']
                    + [f'prev_{c}' for c in CHANGE_COLS] + [f'new_{c}' for c in CHANGE_COLS] + ['delta_mrr_usd']]
                   .sort_values(['date', 'id']).reset_index(drop=True))

print(f'{len(df_cambios_plan):,} cambios de plan de {df_cambios_plan["id"].nunique():,} usuarios')
print(f'Volumen atípico por (día, par de planes): > {MIGRATION_FENCE:.1f} usuarios (cerca de Tukey, k={MASS_TUKEY_K})')
display(pd.crosstab(df_cambios_plan['tipo'], df_cambios_plan['migracion_masiva'], margins=True))
display(df_cambios_plan[df_cambios_plan['migracion_masiva']]
        .groupby(['date', 'prev_package_name', 'new_package_name', 'tipo']).size()
        .rename('usuarios').sort_values(ascending=False).head(10))
display(df_cambios_plan.head(10))

6,182 cambios de plan de 3,906 usuarios
Volumen atípico por (día, par de planes): > 16.0 usuarios (cerca de Tukey, k=3)


migracion_masiva,False,True,All
tipo,,,
downgrade,223,3256,3479
lateral,191,442,633
reajuste_precio,135,42,177
upgrade,977,916,1893
All,1526,4656,6182


date        prev_package_name   new_package_name    tipo     
2026-02-12  Básico              Esencial            downgrade    829
2026-04-28  Básico              Demo EC             downgrade    556
2026-06-23  Esencial            Esencial Paramount  upgrade      406
2026-04-21  Básico              Demo EC             downgrade    351
2026-04-17  Básico              Demo EC             downgrade    305
2026-08-18  Esencial Paramount  Esencial            downgrade    257
2026-04-18  Básico              Demo EC             downgrade    236
2026-06-24  Esencial            Esencial Paramount  upgrade      222
2026-08-19  Esencial Paramount  Esencial            downgrade    215
2026-06-01  Pro                 ZAPPING PRO V2      lateral      205
Name: usuarios, dtype: int64

,date,id,country_iso,tipo,migracion_masiva,same_name,cross_currency,prev_package_id,prev_package_name,prev_package_category,prev_category,prev_currency,prev_original_price,prev_price_paid,prev_mrr_usd,prev_list_price_usd,new_package_id,new_package_name,new_package_category,new_category,new_currency,new_original_price,new_price_paid,new_mrr_usd,new_list_price_usd,delta_mrr_usd
0,2026-01-02,2064909,CL,reajuste_precio,False,True,False,102.0,Zapping Full,NaN,B2C,CLP,18900.0,18900.0,19.063950,19.063950,761,Zapping Full,NaN,B2C,CLP,19900,19900.0,20.072625,20.072625,1.008675
1,2026-01-02,5923819,CL,reajuste_precio,False,True,False,74.0,Zapping Full,NaN,B2C,CLP,17900.0,17900.0,18.055275,18.055275,761,Zapping Full,NaN,B2C,CLP,19900,19900.0,20.072625,20.072625,2.017349
2,2026-01-02,6611055,CL,reajuste_precio,False,True,False,102.0,Zapping Full,NaN,B2C,CLP,18900.0,18900.0,19.063950,19.063950,761,Zapping Full,NaN,B2C,CLP,19900,19900.0,20.072625,20.072625,1.008675
3,2026-01-02,6663250,CL,reajuste_precio,False,True,False,74.0,Zapping Full,NaN,B2C,CLP,17900.0,17900.0,18.055275,18.055275,761,Zapping Full,NaN,B2C,CLP,19900,19900.0,20.072625,20.072625,2.017349
4,2026-01-02,12788842,CL,reajuste_precio,False,True,False,102.0,Zapping Full,NaN,B2C,CLP,18900.0,18900.0,19.063950,19.063950,761,Zapping Full,NaN,B2C,CLP,19900,19900.0,20.072625,20.072625,1.008675
5,2026-01-02,12814032,CL,reajuste_precio,False,True,False,102.0,Zapping Full,NaN,B2C,CLP,18900.0,18900.0,19.063950,19.063950,761,Zapping Full,NaN,B2C,CLP,19900,19900.0,20.072625,20.072625,1.008675
6,2026-01-02,23805012,CL,reajuste_precio,False,True,False,102.0,Zapping Full,NaN,B2C,CLP,18900.0,18900.0,19.063950,19.063950,761,Zapping Full,NaN,B2C,CLP,19900,19900.0,20.072625,20.072625,1.008675
7,2026-01-02,39501902,CL,reajuste_precio,False,True,False,102.0,Zapping Full,NaN,B2C,CLP,18900.0,18900.0,19.063950,19.063950,761,Zapping Full,NaN,B2C,CLP,19900,19900.0,20.072625,20.072625,1.008675
8,2026-01-02,10816310461,CL,reajuste_precio,False,True,False,102.0,Zapping Full,NaN,B2C,CLP,18900.0,18900.0,19.063950,19.063950,761,Zapping Full,NaN,B2C,CLP,19900,19900.0,20.072625,20.072625,1.008675
9,2026-01-03,6051177,CL,reajuste_precio,False,True,False,102.0,Zapping Full,NaN,B2C,CLP,18900.0,18900.0,19.063950,19.063950,761,Zapping Full,NaN,B2C,CLP,19900,19900.0,20.072625,20.072625,1.008675


### 6. Actividad 4: métricas de negocio por cohorte

**Cohortes** consideradas: país, categoría del plan (B2C, BUNDLE, HARDBUNDLE, RESELLER, …), segmento B2B/B2C, plan base (top 8), canal de adquisición (top 8), uso de cupón y antigüedad del usuario.

**Snapshot mensual**: las métricas de stock (usuarios, MRR, ARPPU) se miden el **último día disponible de cada mes**. Septiembre es **parcial** (llega hasta el 07).

Todas las métricas se guardan en **formato largo** (`month, dimension, segment, value, metric`), cómodo para el dashboard (actividad 5). La función `wide()` las pivotea para verlas.

In [16]:
DIMS = ['country_iso', 'category', 'segmento', 'package_grp', 'acquisition_grp', 'coupon_label', 'tenure_bucket']


def by_cohort(frame, func, metric, dims=DIMS, time_col='month', extra_keys=()):
    """Aplica func(groupby) al total y a cada dimensión de cohorte. Devuelve un DataFrame largo."""
    parts = []
    for dim in [None] + list(dims):
        keys = [time_col, *extra_keys] + ([dim] if dim else [])
        val = func(frame.groupby(keys, observed=True)).rename('value').reset_index()
        val['dimension'] = dim or 'total'
        val['segment'] = val[dim].astype(str) if dim else 'Total'
        parts.append(val[[time_col, *extra_keys, 'dimension', 'segment', 'value']])
    out = pd.concat(parts, ignore_index=True)
    out['metric'] = metric
    return out.replace([np.inf, -np.inf], np.nan)


def wide(long, dimension='total', time_col='month', columns='segment'):
    """Pivotea una métrica larga: filas = tiempo, columnas = segmentos de la dimensión."""
    sub = long[long['dimension'] == dimension]
    return sub.pivot_table(index=time_col, columns=columns, values='value', aggfunc='sum', observed=True)


snap_days = user_day.groupby('month')['d'].max()
snap = user_day[user_day['d'].isin(snap_days.values)].copy()
PARTIAL_MONTH = DAYN.to_period('M') if DAYN != DAYN + pd.offsets.MonthEnd(0) else None
print('Snapshots mensuales:', [str(x.date()) for x in snap_days], '| mes parcial:', PARTIAL_MONTH)

Snapshots mensuales: ['2026-01-31', '2026-02-28', '2026-03-31', '2026-04-30', '2026-05-31', '2026-06-30', '2026-07-31', '2026-08-31', '2026-09-07'] | mes parcial: 2026-09


#### 4.1 Usuarios por antigüedad
Usuarios activos en el snapshot mensual según el tiempo transcurrido desde su creación (proxy, supuesto 5).

In [17]:
m_users_tenure = by_cohort(snap, lambda g: g['id'].nunique(), '4.1_usuarios_antiguedad',
                           dims=[d for d in DIMS if d != 'tenure_bucket'], extra_keys=('tenure_bucket',))
display(wide(m_users_tenure, columns='tenure_bucket'))
last_full = snap_days.index[-2] if PARTIAL_MONTH is not None else snap_days.index[-1]
print(f'Distribución de antigüedad por país, {last_full}:')
display(m_users_tenure.query('dimension == "country_iso" and month == @last_full')
        .pivot_table(index='segment', columns='tenure_bucket', values='value', observed=True))

tenure_bucket,<1M,1-3M,3-6M,6-12M,1-2A,>2A
month,,,,,,
2026-01,291,523,891,2905,1768,399
2026-02,540,461,819,2644,2269,420
2026-03,576,611,725,2425,2702,422
2026-04,444,850,650,2263,3005,437
2026-05,303,706,799,2157,3292,426
2026-06,337,531,828,993,1909,460
2026-07,317,464,939,1020,2020,471
2026-08,318,444,732,1167,2015,585
2026-09,315,442,673,1214,2059,596


Distribución de antigüedad por país, 2026-08:


tenure_bucket,<1M,1-3M,3-6M,6-12M,1-2A,>2A
segment,,,,,,
BR,77.0,109.0,176.0,278.0,59.0,45.0
CL,139.0,144.0,184.0,346.0,846.0,429.0
EC,102.0,191.0,372.0,542.0,1110.0,111.0
PE,NaN,NaN,NaN,1.0,NaN,NaN


#### 4.2 Monthly Recurrent Revenue (MRR)
Suma del precio mensual normalizado de todos los planes (base + addons) con `is_mrr_contributor = 1`, activos en el snapshot. Se reporta en USD (consolidado) y en moneda local por país.

In [18]:
m_mrr = by_cohort(snap, lambda g: g['mrr_usd_total'].sum(), '4.2_MRR_usd')
display(wide(m_mrr, 'country_iso').assign(Total=wide(m_mrr)['Total']).round(0))
display(wide(m_mrr, 'category').round(0))

mrr_local = (snap.groupby(['month', 'country_iso', 'currency'])['mrr_local_total'].sum()
             .unstack(['country_iso', 'currency']))
display(mrr_local.round(0))

# Serie diaria (para el dashboard)
m_mrr_daily = by_cohort(user_day, lambda g: g['mrr_usd_total'].sum(), '4.2_MRR_usd_diario', time_col='d')

segment,BR,CL,EC,PE,Total
month,,,,,
2026-01,2687.0,25519.0,31803.0,409.0,60418.0
2026-02,3162.0,26957.0,34503.0,466.0,65088.0
2026-03,3488.0,27442.0,38310.0,494.0,69733.0
2026-04,3844.0,27804.0,32234.0,513.0,64395.0
2026-05,4185.0,28546.0,32290.0,580.0,65601.0
2026-06,4398.0,29034.0,41886.0,110.0,75429.0
2026-07,4593.0,29297.0,43594.0,0.0,77484.0
2026-08,4984.0,30140.0,30212.0,0.0,65336.0
2026-09,5054.0,29998.0,32690.0,0.0,67743.0


segment,B2C,BUNDLE,COUPON,DCB,HARDBUNDLE,HOSPITALITY,PREPAID,REDEEMABLE_COUPON,RESELLER,SIN_CATEGORIA
month,,,,,,,,,,
2026-01,25653.0,18925.0,1376.0,428.0,9878.0,54.0,623.0,NaN,3481.0,NaN
2026-02,29130.0,19317.0,1943.0,469.0,9313.0,54.0,882.0,11.0,3971.0,NaN
2026-03,32842.0,19916.0,1407.0,502.0,9861.0,54.0,1083.0,NaN,4068.0,NaN
2026-04,35601.0,11633.0,1061.0,478.0,10279.0,54.0,1129.0,NaN,4161.0,NaN
2026-05,35917.0,12275.0,1036.0,515.0,10557.0,54.0,917.0,4.0,4326.0,NaN
2026-06,32842.0,6843.0,3199.0,574.0,27956.0,36.0,840.0,18.0,2844.0,278.0
2026-07,33907.0,7296.0,3113.0,646.0,28227.0,36.0,1338.0,0.0,2745.0,175.0
2026-08,35023.0,7542.0,2116.0,611.0,16424.0,36.0,882.0,10.0,2457.0,235.0
2026-09,35005.0,7620.0,2318.0,583.0,18608.0,36.0,887.0,10.0,2443.0,235.0


country_iso       BR          CL    EC               PE     
currency         BRL         CLP   BRL      USD     PEN  USD
month                                                       
2026-01      13488.0  25299551.0  20.0  31799.0  1411.0  NaN
2026-02      15872.0  26725535.0  20.0  34499.0  1608.0  NaN
2026-03      17508.0  27205623.0  20.0  38306.0  1704.0  NaN
2026-04      19297.0  27565270.0  20.0  32230.0  1770.0  NaN
2026-05      21007.0  28300500.0  20.0  32286.0  2002.0  NaN
2026-06      22080.0  28784698.0  20.0  41882.0   379.0  NaN
2026-07      23058.0  29045085.0  20.0  43590.0     NaN  0.0
2026-08      25020.0  29880939.0  20.0  30208.0     NaN  0.0
2026-09      25373.0  29740488.0  20.0  32686.0     NaN  0.0

#### 4.3 Average Revenue Per Paying User (ARPPU)
`ARPPU = MRR / usuarios pagadores`, donde un usuario es pagador si su MRR del día es mayor que 0. Se excluyen planes gratuitos, demos y no contribuyentes.

In [19]:
m_arppu = by_cohort(snap, lambda g: g['mrr_usd_total'].sum() / g['is_paying'].sum(), '4.3_ARPPU_usd')
m_paying = by_cohort(snap, lambda g: g['is_paying'].sum(), 'usuarios_pagadores')
display(wide(m_arppu, 'country_iso').assign(Total=wide(m_arppu)['Total']).round(2))
display(wide(m_arppu, 'segmento').round(2))

segment,BR,CL,EC,PE,Total
month,,,,,
2026-01,6.55,14.41,7.05,10.23,8.97
2026-02,6.74,14.51,7.31,10.13,9.18
2026-03,6.88,14.66,7.80,11.23,9.51
2026-04,6.79,14.76,9.10,10.06,10.65
2026-05,6.94,14.81,9.10,10.95,10.70
2026-06,6.73,14.79,17.87,9.16,15.17
2026-07,6.63,14.71,17.61,0.00,15.02
2026-08,6.71,14.88,12.50,0.00,12.60
2026-09,6.75,14.89,13.26,0.00,12.95


segment,B2B,B2C
month,,
2026-01,6.76,14.64
2026-02,6.75,14.64
2026-03,6.95,14.83
2026-04,7.54,15.03
2026-05,7.63,15.17
2026-06,15.05,15.29
2026-07,14.92,15.12
2026-08,10.13,15.23
2026-09,10.83,15.22


#### 4.4 Monthly Active Users (MAU)
`time_watched` acumula los segundos de consumo de los **últimos 30 días** (supuesto 9), así que en el snapshot de fin de mes indica si el usuario consumió durante ese mes: es la definición estándar de MAU a 30 días. Se mide sobre la misma base que el MRR y el ARPPU (activos en el snapshot) y se reporta el ratio `MAU / suscriptores`, que mide el engagement. Septiembre es parcial: su snapshot (07-09) cubre del 09-08 al 07-09.

Contar `time_watched > 0` en *cualquier* día del mes sobreestimaría el MAU: un usuario que vio algo el 20 de diciembre todavía muestra consumo hasta mediados de enero.

In [20]:
m_mau = by_cohort(snap, lambda g: g['watched'].sum(), '4.4_MAU')
m_subs_month = by_cohort(snap, lambda g: g['id'].nunique(), 'suscriptores_mes')
m_mau_ratio = by_cohort(snap, lambda g: g['watched'].mean(), '4.4_MAU_ratio')
display(pd.DataFrame({'MAU': wide(m_mau)['Total'], 'suscriptores': wide(m_subs_month)['Total'],
                      'MAU/suscriptores': wide(m_mau_ratio)['Total'].round(3)}))
display(wide(m_mau_ratio, 'category').round(3))

# Más de 30 días de consumo en una ventana de 30 días: cuentas con pantallas simultáneas. No afectan al MAU (solo importa > 0)
heavy = user_day['time_watched'] > 30 * 86400
print(f"Usuario-días con más de 30 días de consumo: {heavy.sum():,} ({user_day.loc[heavy, 'id'].nunique():,} usuarios)")

,MAU,suscriptores,MAU/suscriptores
month,,,
2026-01,4392,6777,0.648
2026-02,5010,7153,0.700
2026-03,5252,7461,0.704
2026-04,5210,7649,0.681
2026-05,4889,7683,0.636
2026-06,4023,5058,0.795
2026-07,4149,5231,0.793
2026-08,3983,5261,0.757
2026-09,4021,5299,0.759


segment,B2C,BUNDLE,COUPON,DCB,HARDBUNDLE,HOSPITALITY,PREPAID,REDEEMABLE_COUPON,RESELLER,SIN_CATEGORIA
month,,,,,,,,,,
2026-01,0.938,0.535,0.954,0.967,0.575,0.333,0.977,NaN,0.394,NaN
2026-02,0.958,0.600,0.975,1.000,0.609,0.333,0.984,1.000,0.407,NaN
2026-03,0.961,0.583,0.946,0.971,0.615,0.333,0.976,NaN,0.412,NaN
2026-04,0.963,0.527,0.921,0.971,0.600,0.333,0.988,NaN,0.399,NaN
2026-05,0.967,0.428,0.968,0.974,0.585,0.333,0.970,1.000,0.377,NaN
2026-06,0.962,0.627,0.951,0.976,0.648,1.000,0.981,1.000,0.551,0.600
2026-07,0.975,0.606,0.970,0.978,0.627,0.500,0.977,0.929,0.526,0.667
2026-08,0.968,0.587,0.970,0.913,0.548,0.000,0.970,0.810,0.486,0.500
2026-09,0.967,0.594,0.972,0.933,0.549,0.000,0.985,0.810,0.503,0.500


Usuario-días con más de 30 días de consumo: 32,026 (344 usuarios)


#### 4.5 Vida media por plan
Cada **spell** es un período continuo de un usuario en un mismo plan base (`plan_label` = nombre + país), sin tolerancia a saltos (supuesto 1). El spell termina por cancelación, por un salto **o** por cambio voluntario a otro plan.

- **Eventos masivos.** Si el spell termina por una migración masiva del catálogo (`migracion_masiva`, actividad 3) o por una baja masiva por fin de convenio (`baja_masiva`, actividad 1), se trata como **censura** y no como fin de vida. El usuario no dejó el plan por decisión propia: lo movió Zapping o terminó el contrato del partner.

- **Truncamiento por la izquierda.** Si el spell ya estaba activo el `DAY0`, su vida se mide desde el `created` del plan, y en Kaplan–Meier entra al riesgo recién en `DAY0` (*delayed entry*). Así se evita sesgar la muestra hacia planes antiguos.
- **Censura por la derecha.** Los spells que siguen activos el `DAYN` no tienen fin observado.
- Se reporta la **mediana de Kaplan–Meier** (NaN si la supervivencia nunca baja del 50% en lo observado), la **vida media restringida (RMST) a 365 días** y, como referencia ingenua, el promedio de los spells cerrados.

In [21]:
sp_plan = build_spells(base_day, ['id', 'plan_label'])
sp_plan = sp_plan.merge(base_day[['id', 'd', 'created_dt', 'package_category', 'country_iso']],
                        left_on=['id', 'start'], right_on=['id', 'd'], how='left')
plan_created = sp_plan['created_dt'].dt.normalize()
sp_plan['true_start'] = np.where(sp_plan['left_censored'] & (plan_created < sp_plan['start']),
                                 plan_created, sp_plan['start'])
sp_plan['true_start'] = pd.to_datetime(sp_plan['true_start'])
sp_plan['duration'] = (sp_plan['end'] - sp_plan['true_start']).dt.days + 1
sp_plan['entry'] = (sp_plan['start'] - sp_plan['true_start']).dt.days
mig_idx = pd.MultiIndex.from_frame(df_cambios_plan.loc[df_cambios_plan['migracion_masiva'], ['id', 'date']])
sp_plan['ended_by_migration'] = pd.MultiIndex.from_arrays(
    [sp_plan['id'], sp_plan['end'] + pd.Timedelta(days=1)]).isin(mig_idx)
mass_churn_idx = pd.MultiIndex.from_frame(
    df_altas_bajas.loc[(df_altas_bajas['event_type'] == 'baja') & (df_altas_bajas['baja_masiva'] == True), ['id', 'date']])
sp_plan['ended_by_mass_churn'] = pd.MultiIndex.from_arrays(
    [sp_plan['id'], sp_plan['end'] + pd.Timedelta(days=1)]).isin(mass_churn_idx)
sp_plan['event'] = (~sp_plan['right_censored']
                    & ~sp_plan['ended_by_migration'] & ~sp_plan['ended_by_mass_churn'])
print(f"Spells censurados por migración masiva: {sp_plan['ended_by_migration'].sum():,} | "
      f"por baja masiva (fin de convenio): {sp_plan['ended_by_mass_churn'].sum():,}")


def plan_life(frame, group_col, min_spells=30, curves=None):
    """Resumen de vida por grupo. Si `curves` es una lista, agrega ahí la curva de supervivencia de cada grupo."""
    rows = []
    for name, g in frame.groupby(group_col):
        if len(g) < min_spells:
            continue
        try:
            kmf = KaplanMeierFitter().fit(g['duration'], event_observed=g['event'], entry=g['entry'])
            metodo = 'KM con entrada tardía'
        except StatError:
            # Riesgo inicial demasiado pequeño para el truncamiento: se ajusta sin entrada tardía
            kmf = KaplanMeierFitter().fit(g['duration'], event_observed=g['event'])
            metodo = 'KM sin truncamiento'
        median = kmf.median_survival_time_
        if curves is not None:
            sf = kmf.survival_function_.iloc[:, 0]
            curves.append(pd.DataFrame({'agrupacion': group_col, 'grupo': name, 'dias': sf.index.to_numpy(),
                                        'supervivencia': sf.to_numpy(), 'metodo': metodo}))
        rows.append({group_col: name, 'spells': len(g), 'usuarios': g['id'].nunique(),
                     'pct_censurado': round(1 - g['event'].mean(), 3),
                     'mediana_km_dias': median if np.isfinite(median) else np.nan,
                     'rmst_365_dias': restricted_mean_survival_time(kmf, t=365),
                     'media_cerrados_dias': g.loc[g['event'] & ~g['left_censored'], 'duration'].mean(),
                     'metodo': metodo})
    out = pd.DataFrame(rows).set_index(group_col).sort_values('spells', ascending=False)
    out['mediana_km_meses'] = out['mediana_km_dias'] / 30.44
    out['rmst_365_meses'] = out['rmst_365_dias'] / 30.44
    return out.round(1)


km_curves = []
df_vida_plan = plan_life(sp_plan, 'plan_label', curves=km_curves)
df_vida_categoria = plan_life(sp_plan, 'package_category', curves=km_curves)
df_km_curvas = pd.concat(km_curves, ignore_index=True)  # curvas de Kaplan–Meier, para el dashboard
display(df_vida_plan)
display(df_vida_categoria)

Spells censurados por migración masiva: 4,614 | por baja masiva (fin de convenio): 2,621


,spells,usuarios,pct_censurado,mediana_km_dias,rmst_365_dias,media_cerrados_dias,metodo,mediana_km_meses,rmst_365_meses
plan_label,,,,,,,,,
Básico (EC),4642,4387,0.8,67.0,157.3,19.0,KM con entrada tardía,2.2,5.2
Esencial (EC),2604,1462,0.6,119.0,181.1,40.3,KM con entrada tardía,3.9,5.9
Nacional (CL),1684,1347,0.5,80.0,145.4,53.4,KM con entrada tardía,2.6,4.8
Pro (EC),1589,1191,0.5,90.0,163.3,48.8,KM con entrada tardía,3.0,5.4
Demo EC (EC),1511,1511,1.0,NaN,356.5,13.3,KM con entrada tardía,NaN,11.7
Esencial Paramount (EC),1040,715,0.8,NaN,248.1,28.1,KM con entrada tardía,NaN,8.2
Futbol (CL),999,817,0.5,91.0,151.3,59.4,KM con entrada tardía,3.0,5.0
Zapping Lite Plus (BR),978,809,0.6,268.0,215.6,46.0,KM con entrada tardía,8.8,7.1
Plus (EC),760,567,0.5,93.0,139.8,48.6,KM con entrada tardía,3.1,4.6


,spells,usuarios,pct_censurado,mediana_km_dias,rmst_365_dias,media_cerrados_dias,metodo,mediana_km_meses,rmst_365_meses
package_category,,,,,,,,,
bundle,6426,4219,0.9,568.0,270.1,50.7,KM con entrada tardía,18.7,8.9
hardbundle,5707,1868,0.6,112.0,149.2,35.1,KM con entrada tardía,3.7,4.9
b2c,4749,3401,0.5,79.0,140.8,56.0,KM con entrada tardía,2.6,4.6
prepaid,476,235,0.1,30.0,36.7,32.6,KM con entrada tardía,1.0,1.2
dcb,92,79,0.5,122.0,126.4,58.9,KM con entrada tardía,4.0,4.2
b2c-yearly,38,38,1.0,NaN,365.0,NaN,KM con entrada tardía,NaN,12.0


#### 4.6 Tasa de churn a 1M, 3M y 6M
**Definición:** de los usuarios activos el primer día del mes *M*, el porcentaje que tuvo una baja antes de *M + k meses*, con k = 1, 3, 6, aunque después haya vuelto (supuesto 1). Solo se calcula cuando la ventana completa cabe en los datos (*M + k* `<= DAYN`), así que el churn a 6M solo existe para enero, febrero y marzo. Las cohortes usan los atributos del usuario en *M*.

Como las bajas masivas B2B (término de convenios, p.ej. junio en EC) dominan el indicador, también se calcula el **churn orgánico**, que excluye las bajas con `baja_masiva = True`.

Además se muestra una **matriz de retención** de los usuarios nuevos, por mes de su primera alta.

In [22]:
sp_churn = sp_user.copy()
mass_idx = pd.MultiIndex.from_frame(df_altas_bajas.loc[df_altas_bajas['baja_masiva'] == True, ['id', 'date']])
sp_churn['baja_masiva'] = pd.MultiIndex.from_arrays(
    [sp_churn['id'], sp_churn['end'] + pd.Timedelta(days=1)]).isin(mass_idx)
ud_sorted = user_day.sort_values('d')
churn_parts = []
for month in snap_days.index:
    m_start = month.to_timestamp()
    base = sp_churn[(sp_churn['start'] <= m_start) & (sp_churn['end'] >= m_start)][['id', 'end', 'baja_masiva']].copy()
    base['ref_date'] = m_start
    base = pd.merge_asof(base.sort_values('ref_date'), ud_sorted, left_on='ref_date', right_on='d',
                         by='id', direction='backward')
    for k in (1, 3, 6):
        horizon = m_start + pd.DateOffset(months=k)
        if horizon > DAYN:
            continue
        churned = base['end'] < horizon
        churn_parts.append(base.assign(cohort_month=month, horizonte=f'{k}M', churned=churned,
                                       churned_organico=churned & ~base['baja_masiva']))
churn_base = pd.concat(churn_parts, ignore_index=True)

m_churn = by_cohort(churn_base, lambda g: g['churned'].mean(), '4.6_churn', time_col='cohort_month',
                    extra_keys=('horizonte',))
m_churn_org = by_cohort(churn_base, lambda g: g['churned_organico'].mean(), '4.6_churn_organico',
                        time_col='cohort_month', extra_keys=('horizonte',))
display(pd.concat({'churn total (%)': wide(m_churn, time_col='cohort_month', columns='horizonte'),
                   'churn orgánico (%)': wide(m_churn_org, time_col='cohort_month', columns='horizonte')},
                  axis=1).mul(100).round(1))
for dim in ['country_iso', 'segmento', 'category']:
    print(f'Churn orgánico 1M (%) por {dim}')
    display((m_churn_org.query('dimension == @dim and horizonte == "1M"')
             .pivot_table(index='cohort_month', columns='segment', values='value') * 100).round(1))

churn total (%)             churn orgánico (%)            
horizonte                 1M    3M    6M                 1M    3M    6M
cohort_month                                                           
2026-01                  7.9  16.2  61.6                7.9  16.2  22.7
2026-02                  6.5  14.8  60.8                6.5  14.8  22.2
2026-03                  8.9  15.9  60.6                8.9  15.9  24.3
2026-04                  8.1  51.4   NaN                8.1  16.4   NaN
2026-05                  7.6  51.7   NaN                7.6  17.5   NaN
2026-06                 42.3  51.7   NaN                8.2  17.6   NaN
2026-07                 12.9   NaN   NaN               12.9   NaN   NaN
2026-08                 13.7   NaN   NaN               13.7   NaN   NaN

Churn orgánico 1M (%) por country_iso


segment,BR,CL,EC,PE
cohort_month,,,,
2026-01,10.3,11.6,6.1,26.0
2026-02,7.6,9.5,5.1,16.7
2026-03,12.9,10.5,7.7,28.1
2026-04,8.4,9.5,7.3,19.6
2026-05,9.5,9.5,6.5,12.1
2026-06,8.2,8.6,7.3,66.7
2026-07,7.7,10.4,15.7,95.2
2026-08,8.2,10.9,17.6,0.0


Churn orgánico 1M (%) por segmento


segment,B2B,B2C
cohort_month,,
2026-01,6.2,12.3
2026-02,4.7,11.1
2026-03,6.6,13.9
2026-04,5.5,13.0
2026-05,4.4,13.6
2026-06,5.7,13.1
2026-07,12.4,13.3
2026-08,12.2,15.2


Churn orgánico 1M (%) por category


segment,B2C,BUNDLE,COUPON,DCB,HARDBUNDLE,HOSPITALITY,PREPAID,REDEEMABLE_COUPON,RESELLER,SIN_CATEGORIA
cohort_month,,,,,,,,,,
2026-01,9.7,1.0,36.6,16.7,18.9,0.0,53.3,100.0,2.0,NaN
2026-02,9.2,1.1,19.7,24.1,13.2,0.0,60.5,NaN,3.0,NaN
2026-03,10.9,2.4,24.6,9.4,16.6,0.0,67.2,100.0,5.1,NaN
2026-04,9.5,2.0,24.2,11.8,14.4,0.0,75.0,NaN,2.3,NaN
2026-05,9.5,1.7,36.3,5.9,11.1,0.0,78.8,NaN,1.8,NaN
2026-06,10.0,2.4,30.5,17.9,14.0,33.3,74.6,100.0,1.3,NaN
2026-07,9.0,9.9,31.6,7.1,15.5,0.0,72.2,87.5,5.6,0.0
2026-08,9.2,9.7,39.7,15.2,15.6,0.0,83.9,86.7,3.0,16.7


In [23]:
# Matriz de retención de usuarios nuevos (primera alta dentro de la ventana)
new_users = (df_altas_bajas.query('event_type == "alta" and es_primera_alta')
             .groupby('id')['date'].min().dt.to_period('M').rename('cohort'))
presence = user_day[['id', 'month']].drop_duplicates().merge(new_users.reset_index(), on='id')
presence['k'] = (presence['month'] - presence['cohort']).apply(lambda x: x.n)
df_retencion_nuevos = (presence[presence['k'] >= 0].groupby(['cohort', 'k'])['id'].nunique()
                       .unstack('k')
                       .pipe(lambda t: t.div(t[0], axis=0)))
df_retencion_nuevos.insert(0, 'usuarios_nuevos', new_users.value_counts().sort_index())
display(df_retencion_nuevos.round(3))

k,usuarios_nuevos,0,1,2,3,4,5,6,7,8
cohort,,,,,,,,,,
2026-01,288,1.0,0.997,0.892,0.802,0.729,0.688,0.674,0.646,0.618
2026-02,478,1.0,0.996,0.847,0.774,0.722,0.692,0.651,0.619,NaN
2026-03,585,1.0,0.976,0.809,0.684,0.632,0.614,0.559,NaN,NaN
2026-04,434,1.0,0.977,0.820,0.721,0.666,0.615,NaN,NaN,NaN
2026-05,305,1.0,0.980,0.839,0.685,0.636,NaN,NaN,NaN,NaN
2026-06,336,1.0,0.964,0.789,0.652,NaN,NaN,NaN,NaN,NaN
2026-07,320,1.0,0.984,0.722,NaN,NaN,NaN,NaN,NaN,NaN
2026-08,321,1.0,0.981,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-09,65,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
# Tabla consolidada de métricas (formato largo) para el dashboard
df_metricas = pd.concat([m_users_tenure.rename(columns={'tenure_bucket': 'sub_segment'}),
                         m_mrr, m_arppu, m_paying, m_mau, m_subs_month, m_mau_ratio,
                         m_churn.rename(columns={'cohort_month': 'month', 'horizonte': 'sub_segment'}),
                         m_churn_org.rename(columns={'cohort_month': 'month', 'horizonte': 'sub_segment'})],
                        ignore_index=True)
df_metricas['month'] = df_metricas['month'].astype(str)
df_metricas['sub_segment'] = df_metricas['sub_segment'].astype(str).replace('nan', np.nan)
print(df_metricas.groupby('metric').size())

metric
4.1_usuarios_antiguedad    1658
4.2_MRR_usd                 365
4.3_ARPPU_usd               365
4.4_MAU                     365
4.4_MAU_ratio               365
4.6_churn                   666
4.6_churn_organico          666
suscriptores_mes            365
usuarios_pagadores          365
dtype: int64


### 7. Validaciones

In [25]:
# 1) Identidad de flujo: activos(d) = activos(d-1) + altas(d) - bajas(d)
flow = daily_events['usuarios_activos'].diff().iloc[1:]
assert (flow == daily_events['altas_netas'].iloc[1:]).all(), 'La identidad de flujo diario no cuadra'

# 2) Censura: sin altas en DAY0 ni bajas fuera de la ventana
assert not ((df_altas_bajas['event_type'] == 'alta') & (df_altas_bajas['date'] == DAY0)).any()
assert (df_altas_bajas.loc[df_altas_bajas['event_type'] == 'baja', 'date'] <= DAYN).all()

# 3) Cada cambio de plan tiene al usuario con plan base en d-1 y en d
base_idx = pd.MultiIndex.from_frame(base_day[['id', 'd']])
assert pd.MultiIndex.from_arrays([df_cambios_plan['id'], df_cambios_plan['date']]).isin(base_idx).all()
assert pd.MultiIndex.from_arrays([df_cambios_plan['id'],
                                  df_cambios_plan['date'] - pd.Timedelta(days=1)]).isin(base_idx).all()

# 4) El MRR del snapshot cuadra con la suma directa sobre el parquet
for month, day in snap_days.items():
    direct = df.loc[df['d'] == day, 'mrr_usd'].sum()
    assert np.isclose(wide(m_mrr).loc[month, 'Total'], direct), month

# 5) Toda reactivación 'dentro_ventana' tiene último plan conocido y gap >= 1
dv = df_reactivaciones[df_reactivaciones['origen'] == 'dentro_ventana']
assert dv['last_package_id'].notna().all() and (dv['gap_days'] >= 1).all()

# 6) Cuadre altas: primeras altas + reactivaciones = total de altas de servicio
assert (df_altas_bajas['event_type'] == 'alta').sum() == \
       df_altas_bajas.query('event_type == "alta" and es_primera_alta').shape[0] + len(df_reactivaciones)
# 7) El MAU no supera la base del snapshot, y time_watched es un valor por usuario (igual en todas sus filas del día)
assert (wide(m_mau)['Total'] <= wide(m_subs_month)['Total']).all()
assert (df.groupby(['id', 'd'])['time_watched'].nunique() == 1).all()
print('Todas las validaciones OK')

Todas las validaciones OK


### 8. Exportación para el dashboard

In [26]:
def export(frame, name):
    out = frame.copy()
    out.columns = out.columns.map(str)
    for c in out.columns:
        if isinstance(out[c].dtype, pd.PeriodDtype):
            out[c] = out[c].astype(str)
        elif isinstance(out[c].dtype, pd.CategoricalDtype):
            out[c] = out[c].astype(str)
    out.to_parquet(OUT_DIR / f'{name}.parquet', index=isinstance(out.index, pd.DatetimeIndex) or out.index.name is not None)


for name, frame in {'altas_bajas': df_altas_bajas, 'reactivaciones': df_reactivaciones,
                    'cambios_plan': df_cambios_plan, 'eventos_diarios': daily_events,
                    'user_day': user_day, 'metricas': df_metricas, 'mrr_diario': m_mrr_daily,
                    'vida_plan': df_vida_plan, 'vida_categoria': df_vida_categoria, 'km_curvas': df_km_curvas,
                    'retencion_nuevos': df_retencion_nuevos}.items():
    export(frame, name)
print(sorted(p.name for p in OUT_DIR.glob('*.parquet')))

['altas_bajas.parquet', 'cambios_plan.parquet', 'eventos_diarios.parquet', 'km_curvas.parquet', 'metricas.parquet', 'mrr_diario.parquet', 'reactivaciones.parquet', 'retencion_nuevos.parquet', 'user_day.parquet', 'vida_categoria.parquet', 'vida_plan.parquet']
